In [ ]:
!python --version

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [ ]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [ ]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


In [ ]:
np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
def loop_program_psi_samples(program:str='TCGA', ipsi:Any=None, force:bool=False, verbose:bool=True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

    df_cases, df_subt, df_prof = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if isinstance(ipsi, int):
        lista = [ipsi]
    else:
        lista = np.arange(len(df_psi))

    df_list_cases = []
    df_list_samples = []
    for ipsi in lista:
        row = df_psi.iloc[ipsi]
        pid = row.pid
        primary_site = row.primary_site

        print(ipsi, pid, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)


        if df_cases.empty:
            print("No cases found for PID:", pid)
            continue

        df_list_cases.append(df_cases)

        for isubt, row in df_subt.iterrows():
            print(isubt, end='')
            subtype_global = row.subtype_global
            tumor_class = row.tumor_class
            subtype_tissue = row.subtype_tissue

            s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}'"
            s_case = title_replace(s_case)

            df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                                          tumor_class=tumor_class, subtype_tissue=subtype_tissue, s_case=s_case,
                                                          batch_size=200, force=force, verbose=verbose)
            
            if df_samples.empty:
                print("No samples found for PID:", pid)
                continue


            df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]

            if df2.empty:
                print("No samples having non-blood types for PID:", pid)
                continue

            sample_ids=list(df_samples.sample_id)

            print("\nGetting mutations", end=' ')

            dff, _ = gdc.get_df_mut_transform_mutation_table(study_id=pid, s_case=s_case, sample_ids=sample_ids, force=force, verbose=verbose)

            df_list_samples.append(dff)


    df_all_cases = pd.concat(df_list_cases, ignore_index=True)
    df_all_cases = df_all_cases.drop_duplicates()
    df_all_cases = df_all_cases.reset_index(drop=True)

    df_all_samples = pd.concat(df_list_samples, ignore_index=True)
    df_all_samples = df_all_samples.drop_duplicates()
    df_all_samples = df_all_samples.reset_index(drop=True)

    return df_psi, df_all_cases, df_all_samples



In [ ]:
force=False
verbose=False

df_psi, df_all_cases, df_all_samples = loop_program_psi_samples(program='TCGA', ipsi=None, force=force, verbose=verbose)

print("\n----------- end ------------\n")
print(len(df_all_samples))
